# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-lazyguy/ML-flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
import pandas as pd
import numpy as np

def build_feature_vector(df: pd.DataFrame, cutoff_timestamp: str) -> pd.DataFrame:
    """
    Builds the feature vector enforcing strict temporal boundary control to prevent leakage.
    """
    # 1. Enforce temporal boundary (filtering events before prediction time)
    df = df[df['event_timestamp'] <= cutoff_timestamp].copy()

    features = pd.DataFrame(index=df.index)

    # 2. Numerical Features & Transformations
    features['account_age_days'] = (
        pd.to_datetime(cutoff_timestamp) - pd.to_datetime(df['created_at'])
    ).dt.days.fillna(0)

    features['log_prior_transactions_count'] = np.log1p(
        df['prior_transactions_count'].fillna(0).clip(lower=0)
    )

    features['avg_session_duration_sec'] = (
        df['total_session_sec'] / np.maximum(df['total_sessions'], 1)
    ).fillna(0)

    # 3. Categorical Handling (One-Hot / Target-Agnostic Frequency Encoding)
    device_dummies = pd.get_dummies(df['device_type'], prefix='device', drop_first=True)
    features = pd.concat([features, device_dummies], axis=1)

    # 4. Binary Flags & Missing Value Handling
    features['has_verified_email'] = df['email_verified'].astype(int)
    features['profile_incomplete_flag'] = df['profile_completion_pct'].isna().astype(int)
    features['profile_completion_pct'] = df['profile_completion_pct'].fillna(0.0)

    return features

# Example execution check
# df_raw = pd.read_parquet('raw_events.parquet')
# X = build_feature_vector(df_raw, cutoff_timestamp='2026-08-01 00:00:00')
# print(f"Feature matrix shape: {X.shape}")

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
# Code check: Summarize feature inventory, missing values, and dtypes
def inspect_feature_notes(X: pd.DataFrame):
    """
    Prints a data quality check for the engineered feature matrix.
    """
    summary = pd.DataFrame({
        'Dtype': X.dtypes,
        'Missing_Count': X.isna().sum(),
        'Missing_Pct': (X.isna().mean() * 100).round(2),
        'Unique_Values': X.nunique(),
        'Min_Val': X.min(numeric_only=True).round(2),
        'Max_Val': X.max(numeric_only=True).round(2)
    })

    print("=== FEATURE VECTOR SCHEMA & NULL CHECK ===")
    print(summary)

    assert X.isna().sum().sum() == 0, "ERROR: Unhandled missing values remain in feature matrix!"
    print("\n✓ Check Passed: All missing values properly imputed.")

# Execute check
# inspect_feature_notes(X)

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
def run_leakage_hunt(df_raw: pd.DataFrame, X: pd.DataFrame, y: pd.Series, cutoff_timestamp: str):
    """
    Runs systematic checks for temporal leakage, label leakage, and PII exposure.
    """
    print("=== LEAKAGE & PRIVACY AUDIT ===")

    # 1. Temporal boundary check
    if 'event_timestamp' in df_raw.columns:
        post_cutoff_mask = pd.to_datetime(df_raw['event_timestamp']) > pd.to_datetime(cutoff_timestamp)
        post_cutoff_count = post_cutoff_mask.sum()
        assert post_cutoff_count == 0, f"LEAKAGE FAIL: {post_cutoff_count} records exceed cutoff date!"
        print("✓ Temporal Integrity: Zero post-cutoff records used.")

    # 2. Correlation audit (detect near-identical label copies)
    correlations = X.apply(lambda col: col.corr(y) if col.nunique() > 1 else 0.0).abs()
    suspicious = correlations[correlations > 0.85]

    if not suspicious.empty:
        print(f"⚠️ HIGH CORRELATION WARNING (>0.85):\n{suspicious}")
    else:
        print("✓ Label Leakage Check: No features mirror target variable (>0.85 threshold).")

    # 3. Privacy / PII string check
    pii_terms = ['name', 'email', 'phone', 'ssn', 'address', 'ip_address']
    detected_pii = [col for col in X.columns if any(term in col.lower() for term in pii_terms)]
    assert len(detected_pii) == 0, f"PRIVACY FAIL: Unhashed PII detected: {detected_pii}"
    print("✓ Privacy Audit: Zero raw PII terms found in feature columns.")

# Execute audit
# run_leakage_hunt(df_raw, X, y_target, cutoff_timestamp='2026-08-01 00:00:00')

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
# Code check: Ensure blacklisted/excluded features are absent from feature set
EXCLUDED_FIELDS = [
    'user_email',
    'full_name',
    'post_click_converted_flag',
    'support_ticket_id_after_purchase',
    'total_lifetime_revenue'
]

def verify_exclusions(X: pd.DataFrame, excluded_list: list):
    """
    Asserts that no excluded leakage or PII fields leaked into the feature vector.
    """
    forbidden_present = [col for col in excluded_list if col in X.columns]

    assert len(forbidden_present) == 0, f"EXCLUSION VIOLATION: Found excluded fields in X: {forbidden_present}"
    print("✓ Exclusion Check Passed: All restricted fields were successfully omitted from the feature matrix.")

# Execute check
# verify_exclusions(X, EXCLUDED_FIELDS)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.